# Lab 1.2 &mdash; The Four Building Blocks

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 1 &middot; Module 1 &mdash; Agents vs. Multi-Agent Systems**

### What you'll do
- Write tools that report their failures instead of raising them
- Build short-term memory that compacts instead of growing without bound
- Turn a goal into dependency-ordered steps
- Assemble all four blocks into one small agent over the case file

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Builds on Lab 1.1.** The loop you wrote there is the fourth block; here you build
> the other three and wire them together.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-1-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

## Concept

Each block patches one thing a model cannot do on its own:

| Block | The gap it closes |
|---|---|
| **LLM** | judgement under ambiguity |
| **Memory** | the call is stateless |
| **Tools** | the model cannot read or change anything |
| **Planning** | a goal is not a sequence of steps |

Miss one and you have a pipeline with a model in it &mdash; often the right build, but not an agent.

## Section 1 &mdash; Tools that fail safely

A tool that raises aborts the run. A tool that **returns a description of the failure** hands the
model something it can reason about. Note the docstring: it names the case the tool is for *and*
the case it is not for &mdash; that text is the only thing the model reads when choosing.

In [ ]:
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"    # the agent can recover from this
    return json.dumps({"ref": ref, **record})


def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {"lookup_payment": lookup_payment, "policy_for": policy_for}

In [ ]:
# --- Self-check: Section 1
def _no_raise(fn, *a):
    try:
        return fn(*a), None
    except Exception as exc:
        return None, exc

check("a known reference returns its status",
      lambda: "INSUFFICIENT_FUNDS" in lookup_payment("PMT-1002"))
check("an unknown reference does NOT raise",
      lambda: _no_raise(lookup_payment, "PMT-9999")[1] is None,
      "a raising tool aborts the whole agent run")
check("an unknown reference returns a readable string",
      lambda: isinstance(lookup_payment("PMT-9999"), str) and "PMT-9999" in lookup_payment("PMT-9999"))
check("every tool carries a docstring the model can choose from",
      lambda: all((f.__doc__ or "").strip() for f in TOOLS.values()))

## Section 2 &mdash; Memory that compacts

Unbounded buffer memory is the classic failure: fine in the demo, degraded by week two. Keep the
recent turns verbatim, fold the rest into a summary, and the window stops growing.

In [ ]:
class ShortTermMemory:
    """Recent turns kept verbatim; older ones folded into a running summary."""

    def __init__(self, max_turns: int = 6):
        self.max_turns = max_turns
        self.turns: list[tuple[str, str]] = []
        self.summary = ""

    def add(self, role: str, text: str) -> None:
        self.turns.append((role, text))
        if len(self.turns) > self.max_turns:
            self.compact()

    def compact(self) -> None:
        """Fold all but the most recent turns into `summary`."""
        keep = max(1, self.max_turns // 2)      # keep the recent half, summarise the rest
        older, self.turns = self.turns[:-keep], self.turns[-keep:]
        folded = " ".join(text for _, text in older)
        self.summary = (self.summary + " " + folded).strip()

    def render(self) -> list[tuple[str, str]]:
        """The message list to send: the summary first, then the verbatim turns."""
        head = [("system", "Earlier in this case: " + self.summary)] if self.summary else []
        return head + list(self.turns)

In [ ]:
# --- Self-check: Section 2   (fixtures are built lazily so an unfilled blank cannot crash the cell)
def _short():
    m = ShortTermMemory(max_turns=6)
    for t in ("a", "b", "c"):
        m.add("human", t)
    return m

def _long():
    m = ShortTermMemory(max_turns=6)
    m.add("human", "Investigate PMT-1002.")
    for i in range(20):
        m.add("ai", f"step {i}")
    return m

check("the window stays bounded after 21 turns",
      lambda: len(_long().turns) <= 6,
      "compact() must actually drop turns from self.turns")
check("compaction keeps the earliest instruction somewhere",
      lambda: "PMT-1002" in _long().summary,
      "the oldest turns should be folded into the summary, not discarded")
check("render() puts the summary first",
      lambda: _long().render()[0][0] == "system")
check("a short conversation is left untouched",
      lambda: len(_short().turns) == 3 and _short().summary == "",
      "compact() should only fire once the window is exceeded")

## Section 3 &mdash; Planning: a goal is not a sequence

Decomposition is only half of it. The steps have **dependencies**, and running them out of order
is one of the quieter ways an agent wastes a budget.

In [ ]:
def order_steps(steps: dict[str, list[str]]) -> list[str]:
    """steps maps a step name to the steps it depends on. Return a runnable order.

    Raises ValueError if the dependencies cannot be satisfied (a cycle, or a missing step).
    """
    ordered: list[str] = []
    done: set[str] = set()
    while len(ordered) < len(steps):
        progressed = False
        for name, deps in steps.items():
            if name in done:
                continue
            if all(d in done for d in deps):     # every dependency already ordered
                ordered.append(name)
                done.add(name)
                progressed = True
        if not progressed:
            raise ValueError("cycle or missing dependency in plan")
    return ordered

In [ ]:
# --- Self-check: Section 3
PLAN = {
    "read_payment":  [],
    "read_policy":   ["read_payment"],       # you cannot look up a policy before you know the code
    "decide_action": ["read_payment", "read_policy"],
    "write_note":    ["decide_action"],
}

def _cycles():
    try:
        order_steps({"a": ["b"], "b": ["a"]})
        return False
    except ValueError:
        return True

check("every step follows its dependencies",
      lambda: (lambda o: all(o.index(d) < o.index(n) for n, ds in PLAN.items() for d in ds))(order_steps(PLAN)))
check("the plan starts with the only step that has no dependency",
      lambda: order_steps(PLAN)[0] == "read_payment")
check("all four steps are present exactly once",
      lambda: sorted(order_steps(PLAN)) == sorted(PLAN))
check("a cyclic plan is rejected", lambda: _cycles(),
      "order_steps must raise ValueError rather than loop forever")

## Section 4 &mdash; Assemble the four blocks

Now put them together. Nothing clever &mdash; the point is that you can name which block every line
belongs to.

In [ ]:
class MiniAgent:
    """model + memory + tools + planning, and the loop that binds them."""

    def __init__(self, tools, memory, plan):
        self.tools = tools                # block 3: the hands
        self.memory = memory              # block 2: the state
        self.plan = order_steps(plan)     # block 4: the strategy, in a runnable order
        self.max_steps = 6

    def blocks(self) -> list[str]:
        return ["llm", "memory", "tools", "planning"]

    def run(self, ref: str) -> dict:
        """Walk the plan over one payment. Deterministic -- the model comes in below."""
        self.memory.add("human", f"Investigate {ref}.")
        facts = {}
        for step in self.plan:
            if step == "read_payment":
                facts["payment"] = self.tools["lookup_payment"](ref)
            elif step == "read_policy":
                code_ = json.loads(facts["payment"]).get("reason_code") if facts["payment"].startswith("{") else None
                facts["policy"] = self.tools["policy_for"](code_) if code_ else "no reason code"
            elif step == "decide_action":
                facts["needs_human"] = (json.loads(facts["payment"]).get("reason_code") in NEEDS_HUMAN
                                        if facts["payment"].startswith("{") else False)
            elif step == "write_note":
                self.memory.add("ai", f"{ref}: {facts.get('policy', '')}")
        return facts

In [ ]:
# --- Self-check: Section 4
def _agent():
    return MiniAgent(TOOLS, ShortTermMemory(6), PLAN)

def _out():
    return _agent().run("PMT-1003")

check("the plan was stored in dependency order",
      lambda: _agent().plan[0] == "read_payment",
      "pass the plan through order_steps() in __init__")
check("the agent found the payment", lambda: "LIMIT_BREACH" in _out()["payment"])
check("the agent found the matching policy", lambda: "Treasury" in _out()["policy"])
check("a limit breach is flagged for a human", lambda: _out()["needs_human"] is True)
check("an unknown payment degrades without raising",
      lambda: "no payment found" in MiniAgent(TOOLS, ShortTermMemory(6), PLAN).run("PMT-0000")["payment"])

## Run it for real

Everything above is deterministic. Now let the model do the one part it is actually for &mdash;
turning the facts your tools gathered into a judgement an operator can read.

In [ ]:
if llm_ready():
    try:
        facts = MiniAgent(TOOLS, ShortTermMemory(6), PLAN).run("PMT-1003")
        verdict = ask(
            "You are a payments operations analyst. Using ONLY the facts below, state in two "
            "sentences what happened and what should be done next. If the policy requires a human, "
            "say so explicitly and do not propose acting yourself.\n\n"
            f"PAYMENT: {facts['payment']}\nPOLICY: {facts['policy']}"
        )
        print(verdict)
    except NameError:
        print("(fill in the blanks above, then re-run this cell)")

### Read it

The model never touched the ledger and never chose a policy &mdash; your tools did, deterministically.
It only phrased the judgement. That division is the whole point of the four blocks: put the
verifiable work in code, and leave the model the part that genuinely needs it.

In [ ]:
score()

## Your turn

1. `ShortTermMemory.compact()` concatenates old turns. Replace it with a model-written summary
   (use `ask()`). What did you gain, and what did it cost you in tokens and determinism?
2. `MiniAgent.run` hardcodes the branch per step. Rewrite it as a dict of step handlers. Does
   that make the plan easier to extend, or just harder to read? Argue either way.